In [0]:
# Databricks notebook source

dbutils.widgets.text("config_paths", "")
config_paths_raw = dbutils.widgets.get("config_paths").strip()

if not config_paths_raw:
    raise ValueError("config_paths widget is required")

# Split comma-separated config paths
config_paths = [p.strip() for p in config_paths_raw.split(",") if p.strip()]

# Use your real notebook paths here
BRONZE_NOTEBOOK = "/Users/maddisrikarreddy099@gmail.com/stratus_streaming_platform/02_bronze_ingestion_streaming"
SILVER_NOTEBOOK = "/Users/maddisrikarreddy099@gmail.com/stratus_streaming_platform/03_silver_parse_and_validate_generic"
GOLD_NOTEBOOK   = "/Users/maddisrikarreddy099@gmail.com/stratus_streaming_platform/04_gold_cdc_merge"

NOTEBOOK_TIMEOUT = 0  # 0 means no timeout

import json

def load_config(config_path: str) -> dict:
    config_text = "\n".join(
        row.value for row in spark.read.text(config_path).collect()
    )
    return json.loads(config_text)

def run_stage(stage_name: str, notebook_path: str, config_path: str) -> str:
    print(f"\nRUNNING {stage_name} for {config_path}")
    result = dbutils.notebook.run(
        notebook_path,
        NOTEBOOK_TIMEOUT,
        {"config_path": config_path}
    )
    print(f"{stage_name} RESULT: {result}")
    return result

all_results = []

for config_path in config_paths:
    print("\n" + "#" * 100)
    print(f"STARTING PIPELINE FOR: {config_path}")
    print("#" * 100)

    config = load_config(config_path)

    print(f"ENTITY      : {config['entity_name']}")
    print(f"PRIMARY KEY : {config['primary_key']}")
    print(f"BRONZE PATH : {config['bronze_path']}")
    print(f"SILVER PATH : {config['silver_path']}")
    print(f"GOLD PATH   : {config['gold_path']}")

    entity_result = {
        "config_path": config_path,
        "entity_name": config["entity_name"],
        "bronze": None,
        "silver": None,
        "gold": None,
        "status": "STARTED"
    }

    try:
        entity_result["bronze"] = run_stage("BRONZE", BRONZE_NOTEBOOK, config_path)
        entity_result["silver"] = run_stage("SILVER", SILVER_NOTEBOOK, config_path)
        entity_result["gold"]   = run_stage("GOLD", GOLD_NOTEBOOK, config_path)
        entity_result["status"] = "SUCCESS"
        print(f"PIPELINE COMPLETED FOR: {config['entity_name']}")

    except Exception as e:
        entity_result["status"] = "FAILED"
        entity_result["error"] = str(e)
        print(f"PIPELINE FAILED FOR: {config['entity_name']}")
        print(str(e))

    all_results.append(entity_result)

print("\n" + "=" * 100)
print("FINAL SUMMARY")
print("=" * 100)

for r in all_results:
    print(
        f"ENTITY={r['entity_name']} | STATUS={r['status']} | "
        f"BRONZE={r['bronze']} | SILVER={r['silver']} | GOLD={r['gold']}"
    )

dbutils.notebook.exit(json.dumps(all_results, indent=2))